# Part III-c reported probes: Llama-3.1-8B (Colab A100 80 GB)

The registered prereg's section 2 reports three probes beside the scored verdicts. This runs them for the 8B with
`weight_observer.probes` (master `b4646a9e11e3`): a GPTQ code cache made once by the harness's own path, **verified to
reproduce the registered arms bit for bit before anything else runs**, then the 64-step local search from `gptq_f4`,
the flatness curve around `gptq_f` (38 plans), and the single-matrix sweep at the planned widths (457).

Inputs: the 8B's registered results already in Drive (`MyDrive/tqp-weights/codec/llama3.1-8b/a1f13010bf1d`) and its
committed plans (sha-checked). Outputs: `MyDrive/tqp-weights/probes/llama3.1-8b`, written one measurement at a time; if
Colab disconnects, reconnect to an A100 80 GB and Run all again (the cache is rebuilt, measurements resume).

**Runtime: A100 80 GB, High-RAM.** Expected: ~10 min setup, ~15-20 min cache, then ~4 hours of measurements.

In [ ]:
# 1. GPU: the A100 80 GB (the cost tables peak at ~33 GiB, but the prereg names this product).
import subprocess, torch
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "Runtime > Change runtime type > GPU (A100)"
gib = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"{gib:.1f} GiB"); assert gib > 70, "not the 80 GB A100: reconnect until the first cell shows ~79 GiB"

In [ ]:
# 1b. Google Drive: the registered 8B results (inputs) and the probe outputs.
from google.colab import drive
drive.mount("/content/drive")
import os
REG = "/content/drive/MyDrive/tqp-weights/codec/llama3.1-8b/a1f13010bf1d"
OUT = "/content/drive/MyDrive/tqp-weights/probes/llama3.1-8b"
os.makedirs(f"{OUT}/out", exist_ok=True)
assert os.path.exists(f"{REG}/arms_results.jsonl") and os.path.exists(f"{REG}/codec_costs.jsonl"), "registered results missing"
print("inputs", REG, "| outputs", OUT)

In [ ]:
# 2. The pinned environment (torch and torchvision must match).
!pip -q install "torch==2.8.0" "torchvision==0.23.0" "transformers==4.56.1" "accelerate==1.14.0" safetensors numpy huggingface_hub
!python -c "import torch, transformers; print('pinned:', torch.__version__, transformers.__version__, torch.cuda.get_device_name(0))"

In [ ]:
# 3. The harness at its pinned commit.
import os, subprocess
C = "b4646a9e11e3cb68b6d69666edaf39499b64979a"
if not os.path.exists("/content/tqp"):
    subprocess.run(["git", "clone", "-q", "--filter=blob:none", "--no-checkout",
                    "https://github.com/ahb-sjsu/turboquant-pro", "/content/tqp"], check=True)
    subprocess.run(["git", "-C", "/content/tqp", "sparse-checkout", "set", "--no-cone", "benchmarks/weight_observer"], check=True)
    subprocess.run(["git", "-C", "/content/tqp", "checkout", "-q", C], check=True)
head = subprocess.run(["git", "-C", "/content/tqp", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
assert head == C, head
os.environ.update(PYTHONPATH="/content/tqp/benchmarks", PYTHONUNBUFFERED="1", HF_HUB_DISABLE_XET="1",
                  PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True", TOKENIZERS_PARALLELISM="false")
print("harness", head)
def step(args, label):
    """Run one pinned command, streaming its output here (Colab hides a child's own output)."""
    print(">>", label, flush=True)
    p = subprocess.Popen(["python", "-m", *args], cwd="/content/tqp/benchmarks",
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    lines = []
    for line in p.stdout:
        lines.append(line)
        print(line, end="", flush=True)
    if p.wait():
        raise SystemExit(f"step failed ({p.returncode}): {label}" + chr(10) + "".join(lines[-30:]))


In [ ]:
# 4. The text, restaged and checked against the prereg.
import hashlib
step(["weight_observer.stage_text", "--dest", "/content/wo/text"], "stage_text")
PIN = {"train.txt": "aee724fa58bfbdeb3fc6803297fb6bab27b203d7c40b39ddef9b9770e5d52fe5",
       "test.txt": "696cca6b65a171b0a358a4be6732cdfdf2dd6164a32e20fd70e3c13fc4dfae83"}
for n, h in PIN.items():
    got = hashlib.sha256(open(f"/content/wo/text/{n}", "rb").read()).hexdigest()
    assert got == h, (n, got); print(n, "matches the prereg")

In [ ]:
# 5. The model at the prereg's mirror commit (16.1 GB), manifest checked.
import json
M = "/content/wo/models/llama3.1-8b"
step(["weight_observer.stage_models", "--model-id", "unsloth/Meta-Llama-3.1-8B", "--revision", "e9a141a2091ea561b96483212645a2a05e6f99fc", "--dest", M], "stage_models")
assert json.load(open(f"{M}/STAGED.json"))["revision"] == "e9a141a2091ea561b96483212645a2a05e6f99fc"; print("staged at the prereg's commit")

In [ ]:
# 6. The windows, tokenized once and checked against the registered hashes.
W = "/content/wo/windows/llama3.1-8b"
if not os.path.exists(f"{W}/windows.pt"):
    step(["weight_observer.codec_run", "windows", "--model-path", M, "--text", "/content/wo/text", "--out", W], "windows")
h = json.load(open(f"{W}/hashes.json"))
assert h["calibration_windows"]["sha256"] == "7dcba79b76b0e185ab48d7e27697db01da6345122b6fd0f000a3d64ae9229db3", "calibration windows differ from the prereg"
assert h["evaluation_windows"]["sha256"] == "2eda697bba5e21a5f25b95c50528b3325025ffafc2c7ef020a1e2211af163223", "scored windows differ from the prereg"
print("windows match the registered hashes")

In [ ]:
# 7. The committed plans (sha-checked), then the probes: make, cache, VERIFY (must reproduce the registered arms
#    bit for bit, or everything stops), search, eval, score.
import hashlib
PLANS = "/content/wo/llama3.1-8b.codec_arms.json"
blob = subprocess.run(["git", "-C", "/content/tqp", "show",
                       f"{C}:benchmarks/weight_observer/planned/llama3.1-8b.codec_arms.json"],
                      capture_output=True, check=True).stdout
open(PLANS, "wb").write(blob)  # git show, not the sparse working tree (it lacks planned/)
assert hashlib.sha256(open(PLANS, "rb").read()).hexdigest() == "df98535d971904026b4ad82cb4b50edf939d9bdf0ebdf0405f123d8cd0b81364", "the plans differ from the registered ones"
IN = ["--plans", PLANS, "--costs", f"{REG}/codec_costs.jsonl"]
GPU = ["--model-path", M, "--windows", f"{W}/windows.pt", "--cache", "/content/probe_cache", "--lru-gib", "40"]
step(["weight_observer.probes", "make", *IN, "--out", f"{OUT}/specs.json"], "make")
step(["weight_observer.probes", "cache", "--model-path", M, "--windows", f"{W}/windows.pt",
      "--specs", f"{OUT}/specs.json", "--cache", "/content/probe_cache", "--out", "-"], "cache")
step(["weight_observer.probes", "verify", *GPU, *IN, "--registered", f"{REG}/arms_results.jsonl", "--out", f"{OUT}/out"], "verify")
step(["weight_observer.probes", "search", *GPU, *IN, "--out", f"{OUT}/out"], "search (64 steps)")
step(["weight_observer.probes", "eval", *GPU, "--specs", f"{OUT}/specs.json", "--out", f"{OUT}/out"], "eval (flatness + single)")
step(["weight_observer.probes", "score", *IN, "--registered", f"{REG}/arms_results.jsonl", "--specs", f"{OUT}/specs.json",
      "--probes", f"{OUT}/out", "--out", f"{OUT}/score.json"], "score")

In [ ]:
# 8. Check and hand-back.
v = json.load(open(f"{OUT}/out/verify.json")); s = json.load(open(f"{OUT}/score.json"))
print("verify identical:", v["identical"], "| search steps:", s["search"]["steps"], "accepted:", s["search"]["accepted"])
assert v["identical"] and s["search"]["steps"] == 64
drive.flush_and_unmount()
print("DONE: MyDrive/tqp-weights/probes/llama3.1-8b (G: on the laptop)")